In [82]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tqdm
import copy

In [83]:
from src.utils.normalizer import Normalizer
from typing import Optional, Tuple  


In [84]:
device = "cuda:7"

weight_path = "./models/meta-llama/Llama-2-7b-hf/original_weights/layer_0/self_attn.q_proj.pt"
hessian_diag = weight_path.replace("original_weights", "hessianDiags/seed_0/pajama/128")


weight = torch.load(weight_path, map_location=device)["weight"].to(torch.float32).detach()
hessian_diag = torch.load(hessian_diag, map_location=device)["hessianDiag"].to(torch.float32    )

In [93]:
N,M = 4,8

normalizer, normalized_weight = Normalizer.normalize_init(weight, norm_order=[0,1], zero = [False, False])

In [94]:
importances = torch.abs(normalized_weight 
                        # * hessian_diag.unsqueeze(0)
                        ).view(-1, M)

# get the smallest M-N elements along the last dimension
smallest_idxs = torch.sort(importances, dim=-1).indices[..., :M-N]
mask = torch.ones_like(importances, dtype=torch.bool)
mask.scatter_(1, smallest_idxs, False)
pruned_weight = (normalized_weight.view_as(importances) * mask.float()).view_as(normalized_weight)
pruned_weight

tensor([[-0.0000, -0.0305, -0.0116,  ...,  0.0112,  0.0000, -0.0108],
        [ 0.0138, -0.0000,  0.0093,  ..., -0.0149, -0.0176,  0.0144],
        [-0.0153,  0.0179,  0.0000,  ...,  0.0000,  0.0313, -0.0000],
        ...,
        [ 0.0000,  0.0000, -0.0000,  ...,  0.0096, -0.0283,  0.0114],
        [ 0.0155,  0.0083,  0.0000,  ..., -0.0333, -0.0150, -0.0000],
        [-0.0110, -0.0000,  0.0000,  ...,  0.0244,  0.0217, -0.0133]],
       device='cuda:7')

In [95]:
F.mse_loss(pruned_weight, normalized_weight)

tensor(2.4288e-05, device='cuda:7')

In [96]:
class PermutedPrunedWeightDataFree(nn.Module):
    sparse_idxs: nn.Buffer
    dense_values: nn.Parameter
    permutations: nn.Buffer
    permutation_scales: nn.Parameter
    
    def __init__(self, weight: torch.FloatTensor,
                 N: int = 2, M: int = 4,
                 n_permutations: int = 6
                 ):
        
        super().__init__()
        
        self.original_weight = weight
        self.N = N
        self.M = M
        self.d_out, self.d_in = weight.shape
        
        largest_idxs = torch.sort(weight.abs().view(-1, M), dim=-1).indices[..., -N:] #shape of (d_out*d_in/M, N)
        self.dense_values = nn.Parameter(weight.view(-1, M).gather(1, largest_idxs).view(self.d_out,-1))  # shape of (d_out, N * d_in/M)
        self.register_buffer("sparse_idxs", 
                             (
                            largest_idxs.view(self.d_out, self.d_in // M, N) + torch.arange(0, self.d_in, M, device=weight.device).unsqueeze(0).unsqueeze(-1)
                              ).view(self.d_out,self.d_in//M * N)) # shape of (d_out, d_in/M * N)
        
        permutations = [torch.arange(self.d_in)] + [torch.randperm(self.d_in) for _ in range(n_permutations-1)]
        self.register_buffer("permutations", 
                             torch.stack(permutations, dim=0)) #shape of (n_permutations, d_in)
        
        self.permutation_scales = nn.Parameter(self.compute_permutation_scales()) #shape of (n_permutations, d_in)
        
        
    def reconstruct_weight(self) -> torch.FloatTensor:
        
        sparse_weight = self.reconstruct_sparse_weight()
        permutated_weights = sparse_weight[:, self.permutations]  # shape of (d_out, n_permutations, d_in)
        scaled_weights = torch.einsum("ijk,jk->ik", permutated_weights, self.permutation_scales)  # shape of (d_out, n_permutations, d_in)
        return scaled_weights
        
        
    def reconstruct_sparse_weight(self) -> torch.FloatTensor:
        
        reconstructued_weight = torch.zeros_like(self.original_weight)
        reconstructued_weight[torch.arange(self.d_out).unsqueeze(-1), self.sparse_idxs] = self.dense_values  # shape of (d_out, d_in)
        return reconstructued_weight
    
    @torch.no_grad()
    def compute_permutation_scales(self) -> torch.FloatTensor:
        """
        Compute the permutation scales based on the original weight and the dense values.
        """
        reconstructed_weight = self.reconstruct_sparse_weight()
        
        permutated_weights = reconstructed_weight[:, self.permutations]  # shape of (d_out, n_permutations, d_in)
        
        #use least squares to  compute the optimal scaling factors for each permutation
        optimal_permutations = torch.linalg.lstsq(
            permutated_weights.permute(2,0,1), #shape of (d_in, d_out, n_permutations)
            self.original_weight.T.unsqueeze(2)  # shape of (d_in, d_out,  1)
        )[0].squeeze(2)  # shape of (d_in, n_permutations)
        
        # print(optimal_permutations.shape)
        return optimal_permutations.T  # shape of (n_permutations, d_in)
    
        
    def recon_loss(self) -> torch.FloatTensor:
        """
        Compute the reconstruction loss between the original weight and the reconstructed weight.
        """
        reconstructed_weight = self.reconstruct_weight()
        return F.mse_loss(reconstructed_weight, self.original_weight)
    
    @torch.no_grad()
    def change_permutations(self, new_permutations: torch.Tensor) -> torch.FloatTensor:
        """
        Change the permutations used for reconstruction.
        """
        if new_permutations.shape != self.permutations.shape:
            raise ValueError(f"New permutations must have shape {self.permutations.shape}, but got {new_permutations.shape}.")
        self.register_buffer("permutations", new_permutations)
        # print("previous permutation scales:", self.permutation_scales)
        self.permutation_scales = nn.Parameter(self.compute_permutation_scales())
        # print("new permutation scales computed:", self.permutation_scales)
        return self.recon_loss()

P = PermutedPrunedWeightDataFree(normalized_weight, N=N, M=M, n_permutations=6)
P.recon_loss()
        
        

tensor(2.3947e-05, device='cuda:7', grad_fn=<MseLossBackward0>)

In [97]:
n_greedy_searches = 10
greedy_search_space = 100

lr = 1e-3
n_iters = 100

losses = []

current_best_loss = P.recon_loss().item()
best_state_dict = copy.deepcopy(P.state_dict())
for k in range(greedy_search_space):
    # print("previous loss", P.recon_loss())
    idx = torch.randint(0, P.permutations.shape[0], (1,)).item()
    P.load_state_dict(best_state_dict)  # Reset to the best state found so far
    perms = P.permutations.clone()
    perms[idx] = torch.randperm(P.d_in)
    # print("new permutation", new_permutation)
    # print("current best permutation", current_best_permutation)
    P.change_permutations(perms)
    # print("calling loss fn:", P.recon_loss().item())
    optimizer = torch.optim.Adam(P.parameters(), lr=lr)
    for j in tqdm.tqdm(range(n_iters)):
        optimizer.zero_grad()
        loss = P.recon_loss()
        # tqdm.tqdm.write(f"Iteration {j+1}/{n_iters}, Loss: {loss.item()}")
        loss.backward()
        optimizer.step()
    print(f"Iteration {k+1}/{greedy_search_space}, Loss: {loss.item()}")
    # raise NotImplementedError("Implement permutation change logic if needed.")
    if loss < current_best_loss:
        current_best_loss = loss.item()
        best_state_dict = copy.deepcopy(P.state_dict())
        print(f"Found better permutation at iteration {k+1}, loss: {current_best_loss}")

    
    
    
    
    # #now try to change the permutation 
    # raise NotImplementedError("Implement permutation change logic if needed.")
    
    

100%|██████████| 100/100 [00:08<00:00, 11.30it/s]


Iteration 1/100, Loss: 0.0001996359060285613


100%|██████████| 100/100 [00:08<00:00, 11.31it/s]


Iteration 2/100, Loss: 2.3743243218632415e-05
Found better permutation at iteration 2, loss: 2.3743243218632415e-05


100%|██████████| 100/100 [00:08<00:00, 11.27it/s]


...

Iteration 98/100, Loss: 0.00019939370395150036


100%|██████████| 100/100 [00:09<00:00, 10.90it/s]


Iteration 99/100, Loss: 2.3616943508386612e-05


100%|██████████| 100/100 [00:09<00:00, 10.82it/s]

Iteration 100/100, Loss: 2.3619391868123785e-05


In [ ]:
P.recon_loss()

SyntaxError: invalid syntax (2452869561.py, line 1)

In [80]:
permutations = torch.stack([torch.randperm(a.shape[1]) for _ in range(6)], dim=0)
print(permutations.shape)

a[:,permutations].shape

torch.Size([6, 4096])


torch.Size([4096, 6, 4096])